In [1]:
%pip install selenium requests beautifulsoup4 yt-dlp tqdm pandas

   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ----------------------- ---------------- 1.8/3.2 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 15.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!apt-get update && apt-get install -y ffmpeg

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
import os
import json
import time
import random
from tqdm import tqdm
import yt_dlp

# ====================== CẤU HÌNH ======================
DATASET_DIR = "test_dataset"
POS_DIR = os.path.join(DATASET_DIR, "positive")
NEG_DIR = os.path.join(DATASET_DIR, "negative")

os.makedirs(POS_DIR, exist_ok=True)
os.makedirs(NEG_DIR, exist_ok=True)

TARGET_POS = 150
TARGET_NEG = 150

# ====================== QUERIES TỐI ƯU (RỘNG HƠN) ======================
positive_queries = [
    "factory fire high angle", "factory fire top view", "industrial fire overhead",
    "heavy smoke factory", "black smoke industrial", "white smoke warehouse",
    "visible flame factory", "chemical fire smoke", "burning pallet top view",
    "early smoke fire", "warehouse fire CCTV", "industrial blaze high angle"
]

negative_queries = [
    "factory dust high angle", "industrial steam vapor", "welding sparks smoke",
    "sawdust flying", "factory warning light", "sunlight flare factory",
    "cigarette smoke indoor", "vape smoke", "industrial exhaust steam",
    "woodworking dust", "mist factory", "steam pipe factory"
]

# ====================== DOWNLOAD FUNCTION ======================
def download_video(url, output_path, max_duration=15):
    ydl_opts = {
        'outtmpl': output_path + '.%(ext)s',
        'format': 'best[height<=1080][height>=480]/best',  
        'quiet': True,
        'no_warnings': True,
        'noplaylist': True,
        'download_ranges': lambda info, ydl: [{'start_time': 0, 'end_time': max_duration}],
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            print(f"📥 Tải: {info.get('title', 'Unknown')} | {info.get('duration', 0)}s")
            ydl.download([url])
            return True
    except Exception as e:
        print(f"❌ Lỗi: {e}")
        return False


# ====================== TÌM VIDEO TỪ YOUTUBE (Creative Commons) ======================
def search_youtube_videos(query, max_results=10):
    """Tìm video Creative Commons trên YouTube"""
    search_opts = {
        'quiet': True,
        'extract_flat': True,
        'no_warnings': True,
    }
    with yt_dlp.YoutubeDL(search_opts) as ydl:
        try:
            result = ydl.extract_info(f"ytsearch{max_results}:{query} creative commons", download=False)
            videos = result.get('entries', [])
            return [f"https://www.youtube.com/watch?v={v['id']}" for v in videos if v]
        except:
            return []


# ====================== MAIN ======================
def main():
    print("🚀 BẮT ĐẦU THU THẬP TEST DATASET...\n")
    
    collected_pos = 0
    collected_neg = 0
    
    # POSITIVE
    print("📌 Thu thập POSITIVE (Cháy/Khói)...")
    for query in tqdm(positive_queries):
        if collected_pos >= TARGET_POS:
            break
        print(f"\n🔍 Search: {query}")
        
        urls = search_youtube_videos(query, max_results=8)
        for url in urls:
            if collected_pos >= TARGET_POS:
                break
            filename = f"pos_{collected_pos:04d}"
            output_path = os.path.join(POS_DIR, filename)
            
            if download_video(url, output_path):
                meta = {
                    "id": filename,
                    "class": 1,
                    "query": query,
                    "source": "youtube",
                    "note": "Cần kiểm tra high-angle và cắt 5-15s"
                }
                with open(output_path + ".json", "w", encoding="utf-8") as f:
                    json.dump(meta, f, ensure_ascii=False, indent=2)
                collected_pos += 1
                time.sleep(random.uniform(1, 3))  # Tránh rate limit
    
    # NEGATIVE
    print("\n📌 Thu thập NEGATIVE (Hard Negative)...")
    for query in tqdm(negative_queries):
        if collected_neg >= TARGET_NEG:
            break
        print(f"\n🔍 Search: {query}")
        
        urls = search_youtube_videos(query, max_results=8)
        for url in urls:
            if collected_neg >= TARGET_NEG:
                break
            filename = f"neg_{collected_neg:04d}"
            output_path = os.path.join(NEG_DIR, filename)
            
            if download_video(url, output_path):
                meta = {
                    "id": filename,
                    "class": 0,
                    "query": query,
                    "source": "youtube",
                    "type": "hard_negative"
                }
                with open(output_path + ".json", "w", encoding="utf-8") as f:
                    json.dump(meta, f, ensure_ascii=False, indent=2)
                collected_neg += 1
                time.sleep(random.uniform(1, 3))
    
    print(f"\n🎉 HOÀN TẤT!")
    print(f"   Positive: {collected_pos}/{TARGET_POS}")
    print(f"   Negative: {collected_neg}/{TARGET_NEG}")
    print(f"   Thư mục: {DATASET_DIR}/")

if __name__ == "__main__":
    print("📋 Cài đặt trước khi chạy:")
    print("   pip install yt-dlp tqdm")
    print("   Chạy script với quyền admin nếu cần.\n")
    main()

📋 Cài đặt trước khi chạy:
   pip install yt-dlp tqdm
   Chạy script với quyền admin nếu cần.

🚀 BẮT ĐẦU THU THẬP TEST DATASET...

📌 Thu thập POSITIVE (Cháy/Khói)...


  0%|          | 0/12 [00:00<?, ?it/s]


🔍 Search: factory fire high angle
📥 Tải: Close Up Video Of Fire - Free Stock Creative Commons Video | 20s


ERROR: You have requested downloading the video partially, but ffmpeg is not installed. Aborting


❌ Lỗi: ERROR: You have requested downloading the video partially, but ffmpeg is not installed. Aborting
📥 Tải: Big Gas Fires VFX Stock Footage Collections Now Available | ActionVFX | 128s


...

ERROR: You have requested downloading the video partially, but ffmpeg is not installed. Aborting


❌ Lỗi: ERROR: You have requested downloading the video partially, but ffmpeg is not installed. Aborting


  0%|          | 0/12 [00:08<?, ?it/s]

📥 Tải: Fire and Smoke Dampers | 340s


KeyboardInterrupt: 